# Linear Regression Example (Wine Quality Dataset)

Here it is demonstrated how to use the `LinearRegression` module from the CMOR-438 library to predict wine quality scores.
In this example, the Wine Quality dataset is used to train, test, and evaluate three fitting methods.

**Goal: Predict the quality score of red wine based on its physicochemical properties.**

The Wine Quality dataset has:
- **Samples:** 1,143 red wine observations
- **Features:** 11 physicochemical measurements (alcohol, acidity, sulphates, etc.)
- **Target:** Quality score from 3 to 8 (continuous — treated as regression)

## 1. Setup and Data Loading

Import the necessary modules and load the Wine Quality dataset.
The features are standardised so all inputs are on the same scale before fitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '../_shared')
from linear_regression import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv('../../../data/WineQT.csv').drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']

print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Features: {FEATURE_COLS}")
print(f"Target range: {wine['quality'].min()} – {wine['quality'].max()}")

## 2. Preprocessing

Standardise features to zero mean and unit variance, then split into 80% train / 20% test.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y = wine['quality'].values.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_tr.shape[0]}")
print(f"Test samples:     {X_te.shape[0]}")

## 3. Train — Three Fitting Methods

Three variants of LinearRegression are trained and compared:
- **OLS** — exact closed-form solution via the normal equation
- **Ridge** — OLS with L2 regularisation (alpha=1.0) to prevent overfitting
- **GD** — iterative Gradient Descent over 2,000 steps

In [ ]:
results = {}
for method in ('ols', 'ridge', 'gd'):
    m = LinearRegression(method=method, learning_rate=0.01, n_iterations=2000)
    m.fit(X_tr, y_tr)
    results[method] = {'R2': m.score(X_te, y_te), 'MSE': m.mse(X_te, y_te), 'model': m}
    print(f'{method.upper():>5s} | R²={results[method]["R2"]:.4f}  MSE={results[method]["MSE"]:.4f}')

## 4. Results and Visualisation

Three plots are produced:
- **OLS predicted vs actual** — shows where the model is accurate and where it struggles
- **Ridge predicted vs actual** — comparison to OLS (Ridge adds a small regularisation penalty)
- **GD loss curve** — shows that the gradient descent solver converges steadily

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, method in zip(axes[:2], ('ols', 'ridge')):
    preds = results[method]['model'].predict(X_te)
    ax.scatter(y_te, preds, alpha=0.45, s=18, color='steelblue')
    lo, hi = y_te.min()-0.2, y_te.max()+0.2
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Quality'); ax.set_ylabel('Predicted Quality')
    ax.set_title(f'Linear Reg ({method.upper()})  R²={results[method]["R2"]:.3f}', fontweight='bold')
    ax.legend(fontsize=8)

axes[2].plot(results['gd']['model'].loss_history_, color='darkorange', lw=1.5)
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('MSE Loss')
axes[2].set_title('Gradient Descent Loss Curve', fontweight='bold')
plt.tight_layout(); plt.show()